# SOFA oracle — live demo

SOFA is a powerful but **non-differentiable, runtime-hostile** finite-element solver. Wrapped as a
**Tesseract**, it behaves like a differentiable function the optimizer can call over HTTP:

- `apply`    → the physics value (strain, stress)
- `jacobian` → the gradient w.r.t. the 9 hinge design parameters

The oracle runs in a Docker container; we connect with the `tesseract_core` SDK (`Tesseract.from_url`).

In [2]:
import os
os.environ["JAX_PLATFORMS"] = "cpu"   # CPU only

import sys, pathlib, time
here = pathlib.Path.cwd()
repo = next((p for p in [here, *here.parents] if (p / "nff").is_dir()), here)
sys.path.insert(0, str(repo))

import yaml, numpy as np
from tesseract_core import Tesseract
from nff.sofa import oracle_payload as op

In [3]:
# Connect to the already-running oracle
oracle = Tesseract.from_url("http://localhost:8000")
print("health   :", oracle.health())
print("endpoints:", oracle.available_endpoints)

health   : {'status': 'ok'}
endpoints: ['apply', 'jacobian', 'jacobian_vector_product', 'vector_jacobian_product', 'health', 'abstract_eval']


### Build one hinge design (9 parameters) and the request payload

In [4]:
from nff.sofa.hinge_optimizer import _initial_params, _phys

cfg  = yaml.safe_load(open(repo / "data/configs/sofa/hinge_opt_2face.yaml"))
cs   = op.build_physical_cs(cfg)            # tessellation → physical CentroidalState
phys = _phys(_initial_params(cfg, cs))       # the 9 hinge knobs

payload = op.build_payload(cs, phys, cfg, [0], [1])
payload.update(n_steps=40, skip_secondary_modes=True, mesh_refine=1.0)  # fast demo settings

print("9 design knobs:")
for k in op.PARAM_NAMES:
    print(f"  {k:7s} = {phys[k]:.4f}")

9 design knobs:
  gap     = 0.0046
  s0_top  = 0.0030
  s0_bot  = 0.0030
  s1_top  = 0.0030
  s1_bot  = 0.0030
  bcu_x   = 0.1414
  bcu_y   = 0.0714
  bcl_x   = 0.1414
  bcl_y   = 0.0700


### `apply` — run the physics (one SOFA solve)

In [5]:
t = time.time()
fwd = oracle.apply(payload)
print(f"apply ({time.time()-t:.1f}s)")
print(f"  strain      = {fwd['smooth_principal_strain']*100:.1f} %")
print(f"  peak stress = {fwd['max_von_mises_stress']/1e6:.0f} MPa")

apply (0.9s)
  strain      = 47.7 %
  peak stress = 131 MPa


### Under the hood: the raw HTTP request ↔ response

`oracle.apply(...)` is just an HTTP `POST` to `/apply`. Here is the *same* call sent with `requests`,
so you can see the actual back-and-forth — and the **wrapped** response that the SDK decodes for you.

In [6]:
import requests

# the SDK wraps the payload as {"inputs": ...}; do it by hand to see the raw HTTP
resp = requests.post("http://localhost:8000/apply", json={"inputs": payload})

print("▶ REQUEST :", resp.request.method, resp.request.path_url)
print("   Content-Type:", resp.request.headers["Content-Type"], "| body:", len(resp.request.body), "bytes")
print("◀ RESPONSE:", resp.status_code, resp.reason, "| server:", resp.headers.get("server"),
      "| body:", len(resp.content), "bytes")

raw = resp.json()["max_von_mises_stress"]      # one output, still wrapped
print("\nraw (wrapped) :", raw)
print("decoded value :", raw["data"]["buffer"], "Pa   <- this is what the SDK hands us")

▶ REQUEST : POST /apply
   Content-Type: application/json | body: 1100 bytes
◀ RESPONSE: 200 OK | server: uvicorn | body: 1443 bytes

raw (wrapped) : {'object_type': 'array', 'shape': [], 'dtype': 'float64', 'data': {'buffer': 131327665.12353928, 'encoding': 'json'}}
decoded value : 131327665.12353928 Pa   <- this is what the SDK hands us


### `jacobian` — the gradient (18 finite-difference sims, server-side)

In [7]:
t = time.time()
jac = oracle.jacobian(payload, jac_inputs=op.PARAM_NAMES,
                      jac_outputs=["smooth_principal_strain"])
print(f"jacobian ({time.time()-t:.1f}s)")
for k in op.PARAM_NAMES:
    print(f"  d(strain)/d({k:7s}) = {float(jac['smooth_principal_strain'][k]):8.2f}")

jacobian (6.3s)
  d(strain)/d(gap    ) =    33.29
  d(strain)/d(s0_top ) =  1419.49
  d(strain)/d(s0_bot ) = -1184.44
  d(strain)/d(s1_top ) = -1555.99
  d(strain)/d(s1_bot ) =   983.40
  d(strain)/d(bcu_x  ) =   205.27
  d(strain)/d(bcu_y  ) =  1584.62
  d(strain)/d(bcl_x  ) =    93.59
  d(strain)/d(bcl_y  ) =  1705.97
